In [2]:
#Imports 
import pandas as pd
import re
import numpy as np
import os

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Paths
BRONZE_PATH = "../data/bronze/"
SILVER_PATH = "../data/silver/"



In [3]:
# Load all bronze files
print("Loading bronze files")
cert_bronze = pd.read_csv(os.path.join(BRONZE_PATH, "certifications_bronze.csv"))
clubs_bronze = pd.read_csv(os.path.join(BRONZE_PATH, "clubs_bronze.csv"))
results_bronze = pd.read_csv(os.path.join(BRONZE_PATH, "results_bronze.csv"))

print("\n" + "="*60)
print("BRONZE FILE SHAPES (Before cleaning)")
print("="*60)
print(f"Certifications: {cert_bronze.shape} rows, {cert_bronze.shape[1]} columns")
print(f"Clubs: {clubs_bronze.shape} rows, {clubs_bronze.shape[1]} columns")
print(f"Results: {results_bronze.shape} rows, {results_bronze.shape[1]} columns")

Loading bronze files

BRONZE FILE SHAPES (Before cleaning)
Certifications: (21001, 12) rows, 12 columns
Clubs: (436, 19) rows, 19 columns
Results: (112375, 14) rows, 14 columns


In [4]:
#Checking null values
print("="*60)
print("NULL VALUES IN BRONZE")
print("="*60)

print("\n1. Certifications null counts:")
null_cert = cert_bronze.isnull().sum()
null_cert = null_cert[null_cert > 0]
print(null_cert if len(null_cert) > 0 else "  No null values found")

print("\n2. Clubs null counts:")
null_clubs = clubs_bronze.isnull().sum()
null_clubs = null_clubs[null_clubs > 0]
print(null_clubs if len(null_clubs) > 0 else "  No null values found")

print("\n3. Results null counts:")
null_results = results_bronze.isnull().sum()
null_results = null_results[null_results > 0]
print(null_results if len(null_results) > 0 else "  No null values found")

NULL VALUES IN BRONZE

1. Certifications null counts:
Club                                            780
Code                                            780
Person type                                     780
Gender                                          780
DOB                                            1990
Age                                             780
Mental Handicap (SOB has this certificate)    11188
Parents Consent (SOB has this certificate)    14358
HAP (SOB has this certificate)                15474
Unified Partner (SOB has this certificate)    20818
dtype: int64

2. Clubs null counts:
Address (Street and Number)      2
Zipcode                          3
Province                         6
Country                         74
Participation Games 2015       157
Participation Games 2022       174
Participation Games 2023       136
Participation Games 2024       136
Participation Games 2025       116
dtype: int64

3. Results null counts:
Code                79
Club          

In [5]:
#Cleaning "Certifications"
print("="*60)
print("SILVER: Cleaning Certifications")
print("="*60)

df = cert_bronze.copy()
print(f"Starting shape: {df.shape}")

# Step 1: Standardize Person type 
df['person_type_clean'] = df['Person type'].astype(str).str.lower().str.strip()
print(f"  Person type unique values: {df['person_type_clean'].unique()}")

# Step 2: Standardize Gender (M/F)
df['gender_clean'] = df['Gender'].astype(str).str.upper()
print(f"  Gender unique values: {df['gender_clean'].unique()}")

# Step 3: Convert boolean columns to 0/1 and fill missing with 0
bool_cols = [
    'Mental Handicap (SOB has this certificate)',
    'Parents Consent (SOB has this certificate)',
    'HAP (SOB has this certificate)',
    'Unified Partner (SOB has this certificate)'
]

for col in bool_cols:
    # Convert True/False to 1/0, fill NaN with 0
    df[col] = df[col].fillna(0).astype(int)
    # Also handle string True/False if present
    df[col] = df[col].astype(str).str.lower().map({'true': 1, 'false': 0, '1': 1, '0': 0}).fillna(0).astype(int)
print(f"  Boolean columns converted to 0/1")

# Step 4: Flag missing DOB
df['dob_missing'] = df['DOB'].isnull().astype(int)
print(f"  DOB missing: {df['dob_missing'].sum()} rows ({df['dob_missing'].sum()/len(df)*100:.1f}%)")

# Step 5: Remove rows with null Person type
before = len(df)
df = df[df['person_type_clean'].notna() & (df['person_type_clean'] != 'nan')]
after = len(df)
print(f"  Removed {before - after} rows with missing Person type")

# Step 6: Check null values after cleaning
print(f"\nFinal shape: {df.shape}")
print(f"Null counts after cleaning:")
remaining_nulls = df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print(remaining_nulls if len(remaining_nulls) > 0 else "  No null values remaining")

# Save to silver
output_path = os.path.join(SILVER_PATH, "certifications_silver.csv")
df.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")

SILVER: Cleaning Certifications
Starting shape: (21001, 12)
  Person type unique values: <ArrowStringArray>
[        'athlete',           'coach', 'unified partner',               nan,
       'volunteer',             'vip',           'staff',         'medical',
      'head coach',         'manager',        'security',   'family member',
           'a-hod',           'media',        'as-staff']
Length: 15, dtype: str
  Gender unique values: <ArrowStringArray>
['M', 'F', nan, 'U']
Length: 4, dtype: str
  Boolean columns converted to 0/1
  DOB missing: 1990 rows (9.5%)
  Removed 780 rows with missing Person type

Final shape: (20221, 15)
Null counts after cleaning:
DOB    1210
dtype: int64

Saved to: ../data/silver/certifications_silver.csv


In [6]:
#Validation certifications

df_test = pd.read_csv(os.path.join(SILVER_PATH, "certifications_silver.csv"))
print("Certifications validation:")
print(f"  Shape: {df_test.shape}")
print(f"  Columns: {df_test.columns.tolist()}")
print(f"  Sample person_type values: {df_test['person_type_clean'].head(3).tolist()}")
print(f"  Boolean column sample (HAP): {df_test['HAP (SOB has this certificate)'].head(3).tolist()}")

Certifications validation:
  Shape: (20221, 15)
  Columns: ['Club', 'Code', 'Person type', 'Gender', 'DOB', 'Age', 'Mental Handicap (SOB has this certificate)', 'Parents Consent (SOB has this certificate)', 'HAP (SOB has this certificate)', 'Unified Partner (SOB has this certificate)', 'bronze_timestamp', 'bronze_source', 'person_type_clean', 'gender_clean', 'dob_missing']
  Sample person_type values: ['athlete', 'coach', 'athlete']
  Boolean column sample (HAP): [1, 0, 0]


In [7]:
#Cleaning clubs
print("="*60)
print("SILVER: Cleaning Clubs")
print("="*60)

df = clubs_bronze.copy()
print(f"Starting shape: {df.shape}")

# Step 1: Rename columns for clarity
df = df.rename(columns={
    'Group number': 'club_id',
    'Name': 'club_name',
    'Province': 'region'
})
print(f"  Renamed columns: club_id, club_name, region")

# Step 2: Find all participation columns 
participation_cols = [col for col in df.columns if 'Participation Games' in col]
print(f"  Found {len(participation_cols)} participation year columns")

# Step 3: Convert participation flags to 0/1
for col in participation_cols:
    df[col] = df[col].fillna(0).astype(int)
    # Handle string True/False if present
    df[col] = df[col].astype(str).str.lower().map({'true': 1, 'false': 0, '1': 1, '0': 0}).fillna(0).astype(int)
print(f"  Participation flags converted to 0/1")

# Step 4: Create total_participations column 
df['total_participations'] = df[participation_cols].sum(axis=1)
print(f"  Created total_participations column (range: {df['total_participations'].min()} to {df['total_participations'].max()})")

# Step 5: Check null values after cleaning
print(f"\nFinal shape: {df.shape}")
print(f"Null counts after cleaning:")
remaining_nulls = df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print(remaining_nulls if len(remaining_nulls) > 0 else "  No null values remaining")

# Save to silver
output_path = os.path.join(SILVER_PATH, "clubs_silver.csv")
df.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")

SILVER: Cleaning Clubs
Starting shape: (436, 19)
  Renamed columns: club_id, club_name, region
  Found 9 participation year columns
  Participation flags converted to 0/1
  Created total_participations column (range: 0 to 9)

Final shape: (436, 20)
Null counts after cleaning:
Address (Street and Number)     2
Zipcode                         3
region                          6
Country                        74
dtype: int64

Saved to: ../data/silver/clubs_silver.csv


In [8]:
df_test = pd.read_csv(os.path.join(SILVER_PATH, "clubs_silver.csv"))
print("Clubs validation:")
print(f"  Shape: {df_test.shape}")
print(f"  Region unique values: {df_test['region'].nunique()}")
print(f"  Total participations range: {df_test['total_participations'].min()} to {df_test['total_participations'].max()}")
print(f"  Sample: {df_test[['club_id', 'club_name', 'region', 'total_participations']].head(2)}")

Clubs validation:
  Shape: (436, 20)
  Region unique values: 24
  Total participations range: 0 to 9
  Sample:    club_id   club_name      region  total_participations
0      100  LA PILERIE     Hainaut                     9
1      101         BAM  Luxembourg                     3


In [10]:
#Results
print("="*60)
print("SILVER: Cleaning Results")
print("="*60)

df = results_bronze.copy()
print(f"Starting shape: {df.shape}")

# Step 1: Standardize Gender (Male/Female -> M/F)
df['gender_clean'] = df['Gender'].map({'Male': 'M', 'Female': 'F'})
print(f"  Gender mapping complete")
print(f"  Gender unique values: {df['gender_clean'].unique()}")

# Step 2: Extract numeric rank from Place column
def extract_rank(place):
    if pd.isna(place):
        return None
    match = re.search(r'(\d+)', str(place))
    return int(match.group(1)) if match else None

df['rank_numeric'] = df['Place'].apply(extract_rank)
ranks_found = df['rank_numeric'].notna().sum()
print(f"  Extracted numeric rank: {ranks_found}/{len(df)} rows ({ranks_found/len(df)*100:.1f}%)")

# Step 3: Extract numeric score from Score column
def extract_score(score):
    if pd.isna(score):
        return None
    # Look for first number (integer or decimal)
    match = re.search(r'(\d+(?:\.\d+)?)', str(score))
    return float(match.group(1)) if match else None

df['score_numeric'] = df['Score'].apply(extract_score)
scores_found = df['score_numeric'].notna().sum()
print(f"  Extracted numeric score: {scores_found}/{len(df)} rows ({scores_found/len(df)*100:.1f}%)")

# Step 4: Flag disqualifications
df['is_disqualified'] = df['Summary (all)'].fillna('').astype(str).str.contains('DQ|Disqualified', case=False, na=False).astype(int)
df['is_disqualified'] |= df['Score'].fillna('').astype(str).str.contains('DQ|Disqualified', case=False, na=False).astype(int)
dq_count = df['is_disqualified'].sum()
print(f"  Disqualifications flagged: {dq_count} rows ({dq_count/len(df)*100:.1f}%)")

# Step 5: Extract main sport from Sport column (take part before slash)
def extract_sport(sport):
    if pd.isna(sport):
        return None
    return str(sport).split('/')[0].strip()

df['sport_clean'] = df['Sport'].apply(extract_sport)
print(f"  Sport extraction complete")
print(f"  Unique sports: {df['sport_clean'].nunique()}")

# Step 6: Ensure source_year is integer
df['year'] = df['source_year'].astype(int)
print(f"  Years present: {sorted(df['year'].unique())}")

# Step 7: Remove rows with missing required fields
before = len(df)
df = df[df['Code'].notna() & (df['Code'].astype(str).str.strip() != '')]
df = df[df['sport_clean'].notna() & (df['sport_clean'].astype(str).str.strip() != '')]
df = df[df['Club'].notna() & (df['Club'].astype(str).str.strip() != '')]
after = len(df)
print(f"  Removed {before - after} rows with missing Code, Sport, or Club")

print(f"\nFinal shape: {df.shape}")

# Step 8: Check null values after cleaning
print(f"\nNull counts after cleaning:")
remaining_nulls = df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print(remaining_nulls if len(remaining_nulls) > 0 else "  No critical null values remaining")

# Save to silver
output_path = os.path.join(SILVER_PATH, "results_silver.csv")
df.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")

SILVER: Cleaning Results
Starting shape: (112375, 14)
  Gender mapping complete
  Gender unique values: <ArrowStringArray>
['M', 'F', nan]
Length: 3, dtype: str
  Extracted numeric rank: 81085/112375 rows (72.2%)
  Extracted numeric score: 83243/112375 rows (74.1%)
  Disqualifications flagged: 2241 rows (2.0%)
  Sport extraction complete
  Unique sports: 23
  Years present: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
  Removed 79 rows with missing Code, Sport, or Club

Final shape: (112296, 20)

Null counts after cleaning:
DOB                 60
Place            27351
Score            28789
Summary (all)    11735
gender_clean        11
rank_numeric     31211
score_numeric    29053
dtype: int64

Saved to: ../data/silver/results_silver.csv


In [11]:
df_test = pd.read_csv(os.path.join(SILVER_PATH, "results_silver.csv"))
print("Results validation:")
print(f"  Shape: {df_test.shape}")
print(f"  Years present: {sorted(df_test['year'].unique())}")
print(f"  Gender values: {df_test['gender_clean'].unique()}")
print(f"  Disqualifications: {df_test['is_disqualified'].sum()} rows")
print(f"\nSample of extracted values:")
print(df_test[['Place', 'rank_numeric', 'Score', 'score_numeric', 'sport_clean']].head(3))

Results validation:
  Shape: (112296, 20)
  Years present: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
  Gender values: <ArrowStringArray>
['M', 'F', nan]
Length: 3, dtype: str
  Disqualifications: 2241 rows

Sample of extracted values:
  Place  rank_numeric            Score  score_numeric sport_clean
0   1st           1.0     18m, 28.00cm           18.0   Athletics
1   2nd           2.0      1m, 51.00cm            1.0   Athletics
2   2nd           2.0  0 min, 8.89 sec            0.0   Athletics


In [12]:
print("="*60)
print("SILVER LAYER SUMMARY")
print("="*60)

print("\nFile size comparison (Bronze vs Silver):")
for file in ['certifications', 'clubs', 'results']:
    bronze_size = os.path.getsize(os.path.join(BRONZE_PATH, f"{file}_bronze.csv")) / 1024
    silver_size = os.path.getsize(os.path.join(SILVER_PATH, f"{file}_silver.csv")) / 1024
    print(f"  {file}: Bronze={bronze_size:.1f}KB, Silver={silver_size:.1f}KB, Change={silver_size-bronze_size:.1f}KB")

print("\nRow counts (Bronze vs Silver):")
cert_clean = pd.read_csv(os.path.join(SILVER_PATH, "certifications_silver.csv"))
clubs_clean = pd.read_csv(os.path.join(SILVER_PATH, "clubs_silver.csv"))
results_clean = pd.read_csv(os.path.join(SILVER_PATH, "results_silver.csv"))

print(f"  Certifications: {cert_bronze.shape[0]} -> {cert_clean.shape[0]} (removed {cert_bronze.shape[0] - cert_clean.shape[0]})")
print(f"  Clubs: {clubs_bronze.shape[0]} -> {clubs_clean.shape[0]} (removed {clubs_bronze.shape[0] - clubs_clean.shape[0]})")
print(f"  Results: {results_bronze.shape[0]} -> {results_clean.shape[0]} (removed {results_bronze.shape[0] - results_clean.shape[0]})")

print("\nSILVER LAYER COMPLETE - Ready for Gold Layer")

SILVER LAYER SUMMARY

File size comparison (Bronze vs Silver):
  certifications: Bronze=2558.2KB, Silver=2748.7KB, Change=190.5KB
  clubs: Bronze=73.9KB, Silver=67.0KB, Change=-7.0KB
  results: Bronze=25889.6KB, Silver=28794.9KB, Change=2905.4KB

Row counts (Bronze vs Silver):
  Certifications: 21001 -> 20221 (removed 780)
  Clubs: 436 -> 436 (removed 0)
  Results: 112375 -> 112296 (removed 79)

SILVER LAYER COMPLETE - Ready for Gold Layer
